In [1]:
import torch
import random
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import numpy as np
import pickle
import matplotlib.pyplot as plt
from IPython.display import clear_output
from xFNO import nFNO

random.seed(100)
device = "cpu"

In [2]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)     
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

def test(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    model.eval()
    loss = 0
    with torch.no_grad():   
        for batch, (X,y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss += loss_fn(pred, y).item()
    loss /= num_batches
    return loss

In [3]:
# Load PDE solution
with open('data/KuramotoSivashinsky/KS_N32.pkl', 'rb') as file:
    data = pickle.load(file)
data = torch.tensor(data)
print(data.size())
Nx = data.size()[1]

torch.Size([2001, 32])


In [7]:
# Make datasets, dataloaders
history = 5
dt = 30 # the sequences overlap less if dt is larger. overlap: dt - history
shuffle_flag=True # shuffling helps a lot
training_size = 1024
data_truncated = data[:history*(data.size()[0]//history), :]

## need to reshape these [B, Nx, Nt, 1] where Nx is the spatial resolution and Nt is the temporal one
features = torch.stack([
        torch.reshape(
        data_truncated[i:i+history, :].to(torch.float32).t(), 
            (Nx, history, 1)) for i in range(data_truncated.size()[0]-history-dt)
])

labels = torch.stack([
        torch.reshape(
        data_truncated[dt+i:dt+i+history, :].to(torch.float32).t(), 
            (Nx, history, 1)) for i in range(data_truncated.size()[0]-history-dt)
])

# datasets
if shuffle_flag: 
    idx = torch.randperm(training_size)
else:
    idx = torch.arange(training_size)
training_dataset = TensorDataset(features[idx, :, :, :], labels[idx, :, :, :])
testing_dataset = TensorDataset(features[training_size:, :, :, :], labels[training_size:, :, :, :])

# dataloaders
batch_size = 64
training_dataloader = DataLoader(training_dataset, batch_size=batch_size, shuffle=False)
testing_dataloader = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)


In [8]:
# initialize operator
Nf = 1
Nlifted = 16
Nk_truncated = 10
Nw_truncated = 2
depth = 1
#use_hamming_window = False
neuralop = nFNO(
            (Nx, history, Nf), 
            Nlifted, 
            (Nk_truncated, Nw_truncated), 
            depth
            ).to(device) # shallow is more stable
loss_fn = torch.nn.MSELoss(reduction='mean')
rate = 2e-04
optimizer_neuralop = torch.optim.Adam(neuralop.parameters(), lr=rate)
progress = []

In [ ]:
# train
epochs = 600
for i in range(epochs):
    train(training_dataloader, neuralop, loss_fn, optimizer_neuralop)
    progress += [test(testing_dataloader, neuralop, loss_fn)]
    if i%100==0:
        print(f"Loss: {(progress[-1]):>0.5f}")
        rate *= 0.9
        optimizer_neuralop = torch.optim.Adam(neuralop.parameters(), lr=rate)
    if np.isnan(progress[-1]): 
        print('nan!')
        break
print(f'final learning rate: {rate}')
plt.figure()
plt.plot(progress)
plt.loglog()

Loss: 0.08300
Loss: 0.08067
Loss: 0.07868
Loss: 0.07662
Loss: 0.07450


In [ ]:
# unroll, a priori model
# test predicted timeseries of length 'history'
neuralop.eval()
fig, ax = plt.subplots()
for i in range(data_truncated[training_size:-history-dt, :].size()[0]):
    if i%history == 0:
        uin = torch.reshape(
            data_truncated[training_size+i:training_size+i+history, :].to(torch.float32).t(), 
            (1, Nx, history, 1)
            )
        u_pred = neuralop(uin) # don't extrapolate, recompute prediction
    u_plot = u_pred.detach().numpy().squeeze()[:, i%history]
        
    clear_output(wait=True)
    fig, ax = plt.subplots()
    ax.plot(data_truncated[i+dt+training_size, :])
    ax.plot(u_plot, '--')
    plt.show()
    plt.pause(0.1)

In [9]:
# unroll, a priori model
# test farthest prediction in the future
neuralop.eval()
fig, ax = plt.subplots()
for i in range(data_truncated[training_size:-history-dt, :].size()[0]):
    uin = torch.reshape(
            data_truncated[training_size+i:training_size+i+history, :].to(torch.float32).t(), 
            (1, Nx, history, 1)
        )
    u_pred = neuralop(uin) # don't extrapolate, recompute prediction
    u_plot = u_pred.detach().numpy().squeeze()[:, -1]
        
    clear_output(wait=True)
    fig, ax = plt.subplots()
    ax.plot(data_truncated[i+dt+training_size+history-1, :])
    ax.plot(u_plot, '--')
    plt.show()
    plt.pause(0.1)

KeyboardInterrupt: 